# RAG Agent

In [1]:
import os
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient


load_dotenv()

model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME"),
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),
)

embedding_model = DashScopeEmbeddings(
    model="text-embedding-v4",
    dashscope_api_key=os.environ.get("COMPATIBLE_API_KEY"),
)

client = QdrantClient(host="localhost", port=6333)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name="agentic_rag_survey",
    embedding=embedding_model,
)

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vectorstore.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

tools = [retrieve_context]
prompt = (
    "You have access to a tool that retrieves context from a paper. "
    "Use the tool to help answer user queries."
)

# No Memory

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage


agent_without_memory = create_agent(model, tools, system_prompt=prompt)

queries = ["What is RAG?", "What did I asked?"]
for query in queries:
    for step in agent_without_memory.stream(
        {"messages": [HumanMessage(query)]}, stream_mode="values"):
        step["messages"][-1].pretty_print()

================================ Human Message =================================

What is RAG?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_804ee1bf5b0c4241a8fcd507)
 Call ID: call_804ee1bf5b0c4241a8fcd507
  Args:
    query: What is RAG?
================================= Tool Message =================================
Name: retrieve_context

Source: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-02-05T01:26:00+00:00', 'author': '', 'keywords': '', 'moddate': '2025-02-05T01:26:00+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../example_data/A_SURVEY_ON_AGENTIC_RAG.pdf', 'total_pages': 39, 'page': 0, 'page_label': '1', 'start_index': 781, '_id': 'e984b0d2-381c-4368-81eb-9525475d40c6', '_collection_name': 'agentic_rag_survey'}
Content: outputs. Retri

# Short-term memory

In [3]:
from langgraph.checkpoint.memory import InMemorySaver


agent_with_memory = create_agent(
    model, 
    tools, 
    system_prompt=prompt, 
    checkpointer=InMemorySaver()
)

queries = ["What is RAG?", "What did I asked?"]
config = {"configurable": {"thread_id": "1"}}
for query in queries:
    for step in agent_with_memory.stream(
        {"messages": [HumanMessage(query)]}, config=config, stream_mode="values"):
        step["messages"][-1].pretty_print()

================================ Human Message =================================

What is RAG?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_1b426731fb594e8f88912075)
 Call ID: call_1b426731fb594e8f88912075
  Args:
    query: What is RAG?
================================= Tool Message =================================
Name: retrieve_context

Source: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-02-05T01:26:00+00:00', 'author': '', 'keywords': '', 'moddate': '2025-02-05T01:26:00+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../example_data/A_SURVEY_ON_AGENTIC_RAG.pdf', 'total_pages': 39, 'page': 0, 'page_label': '1', 'start_index': 781, '_id': 'e984b0d2-381c-4368-81eb-9525475d40c6', '_collection_name': 'agentic_rag_survey'}
Content: outputs. Retri

# Manage Memory

- Trim Messages
- Delete messages
- Summarize messages

### Trim Messages

In [4]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Keep only the last few messages to fit context window."""
    messages = state["messages"]

    if len(messages) <= 3:
        return None  # No changes needed

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }

agent = create_agent(
    model,
    tools=tools,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好，我是李明。"}, config)
agent.invoke({"messages": "我35了。"}, config)
agent.invoke({"messages": "我喜欢看恐怖小说。"}, config)
final_response = agent.invoke({"messages": "我叫什么？"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

你叫李明！


### Delete messages

In [5]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    """Remove old messages to keep conversation manageable."""
    messages = state["messages"]
    if len(messages) > 2:
        # remove the earliest two messages
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:2]]}
    return None


agent = create_agent(
    model,
    tools=[],
    system_prompt="Please be concise and to the point.",
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

for event in agent.stream(
    {"messages": [HumanMessage("你好，我叫韩梅梅！")]},
    config,
    stream_mode="values",
):
    print([(message.type, message.content) for message in event["messages"]])

for event in agent.stream(
    {"messages": [HumanMessage("我叫什么？")]},
    config,
    stream_mode="values",
):
    print([(message.type, message.content) for message in event["messages"]])

[('human', '你好，我叫韩梅梅！')]
[('human', '你好，我叫韩梅梅！'), ('ai', '你好，韩梅梅！很高兴认识你。')]
[('human', '你好，我叫韩梅梅！'), ('ai', '你好，韩梅梅！很高兴认识你。'), ('human', '我叫什么？')]
[('human', '你好，我叫韩梅梅！'), ('ai', '你好，韩梅梅！很高兴认识你。'), ('human', '我叫什么？'), ('ai', '你叫韩梅梅！')]
[('human', '我叫什么？'), ('ai', '你叫韩梅梅！')]


### Summarize messages

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig


checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=model,
            max_tokens_before_summary=100,  # Trigger summarization at 100 tokens
            messages_to_keep=4,  # Keep last 4 messages after summary
        )
    ],
    checkpointer=checkpointer,
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}
agent.invoke({"messages": "你好，我叫李明！"}, config)
agent.invoke({"messages": "写一首关于秋天的诗句。"}, config)
agent.invoke({"messages": "那么关于冬天的呢？"}, config)
final_response = agent.invoke({"messages": "我叫什么？"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

你叫李明！😊  
（在之前的对话中，你提到过自己的名字哦～）


# Customizing agent memory

In [ ]:
from langchain.agents import create_agent, AgentState
from langchain.tools import ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver


class CustomAgentState(AgentState):  
    user_id: str

@tool
def get_user_info(runtime: ToolRuntime) -> str:
    """Look up user info."""
    user_id = runtime.state["user_id"]
    return "User is John Smith" if user_id == "user_123" else "Unknown user"

agent = create_agent(
    model,
    [get_user_info],
    state_schema=CustomAgentState,  
    checkpointer=InMemorySaver(),
)

# Custom state can be passed in invoke
for step in agent.stream({
        "messages": [HumanMessage("Hello. What is my name?")],
        "user_id": "user_123"
    }, 
    {"configurable": {"thread_id": "1"}}, stream_mode="values"):
        step["messages"][-1].pretty_print()

================================ Human Message =================================

Hello. What is my name?
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_c82f70f38f3248ebb41e695e)
 Call ID: call_c82f70f38f3248ebb41e695e
  Args:
================================= Tool Message =================================
Name: get_user_info

User is John Smith
================================== Ai Message ==================================

Your name is John Smith.
